## With this, i want to make the general analysis a little bit skinnier

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np

sys.path.insert(0, "/jetfs/home/gbrack/MSC_Thesis/flexpart_tools/src")

from flexpart_tools.read_binary import read_namelists, read_gridini
from flexpart_tools.era5_io import read_met
from flexpart_tools.era5_vertical import calculate_geometry, average_to_flexpart_layers, geometric_altitude

: 

In [2]:
run_dir = Path(
    "/jetfs/scratch/gbrack/flexpart_output/"
    "new_delhi_30d_24h_hourly_releases_bigger_domain"
)

met_dir = Path("/jetfs/shared-data/ECMWF/ERA5_glob_0.5deg_1h")

times_ini, times_fin, heights, lat, lon, age = read_namelists(run_dir)

release = 1
i = release - 1

print("Rückwärtsdauer [Tage]", age / 86400)
print("Anfangszeit [UTC]", times_ini[i])
print("Releasezeit [UTC]", times_fin[i])
print("FLEXPART-Schichten", len(heights) - 1)

Rückwärtsdauer [Tage] 30.0
Anfangszeit [UTC] 2015-04-25 00:00:00
Releasezeit [UTC] 2015-05-25 00:00:00
FLEXPART-Schichten 42


In [3]:
ds_ini = read_gridini(run_dir, times_ini, heights, lat, lon)
s_ini = ds_ini.sensitivity.isel(valid_time=1, drop=True)

weight_sum = s_ini.sum(skipna=False).item()
print("Sum of initial weights:", weight_sum)

Sum of initial weights: 0.8838453127760778


In [4]:
initial_time = pd.Timestamp(times_ini[i])
met_file = met_dir / initial_time.strftime("%Y/%m/EA%Y%m%d%H")
met_ini = read_met(initial_time, met_dir).sel(
    latitude=s_ini.latitude.values,
    longitude=s_ini.longitude.values
)

print(met_ini.sizes["hybrid"])
print(met_ini.sizes["hybrid_half"])

height_full, height_half = calculate_geometry(
    met_ini, met_ini.a_half.values, met_ini.b_half.values
)

137
138


In [23]:
met_ini

<xarray.Dataset> Size: 163MB
Dimensions:     (hybrid: 137, latitude: 230, longitude: 640)
Coordinates:
    time        datetime64[ns] 8B 2015-04-25
    step        timedelta64[ns] 8B 00:00:00
  * hybrid      (hybrid) float64 1kB 1.0 2.0 3.0 4.0 ... 134.0 135.0 136.0 137.0
  * latitude    (latitude) float64 2kB -30.0 -29.5 -29.0 ... 83.5 84.0 84.5
    valid_time  datetime64[ns] 8B 2015-04-25
    number      int64 8B 0
  * longitude   (longitude) float64 5kB -150.0 -149.5 -149.0 ... 169.0 169.5
Data variables:
    t           (hybrid, latitude, longitude) float32 81MB 185.7 185.8 ... 264.9
    q           (hybrid, latitude, longitude) float32 81MB 1.417e-06 ... 0.00172
    sp          (latitude, longitude) float32 589kB 1.013e+05 ... 1.012e+05
    z           (latitude, longitude) float32 589kB -2.759 1.569 ... -0.2632
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-06T07:55 GRIB to CDM+CF via cfgrib-0.9.1...

In [12]:
cp, Lv, g = 1004.6, 2501000.0, 9.81

ground_altitude = geometric_altitude(
    met_ini.z.transpose("latitude", "longitude").values.astype("float64")
)

energy_fields = {
    "cpT": cp * met_ini.t.transpose("hybrid", "latitude", "longitude"),
    "Lvq": Lv * met_ini.q.transpose("hybrid", "latitude", "longitude"),
    "gh": g * (height_full + ground_altitude)
}

initial_raw = {}

for name, values in energy_fields.items():
    layer_mean = average_to_flexpart_layers(values, height_half, s_ini.height.values)
    field = s_ini.copy(data=layer_mean)
    initial_raw[name] = (s_ini * field).sum(skipna=False).item()

initial_raw["MSE"] = sum(initial_raw.values())

initial_budget = pd.Series(initial_raw, name="raw_Jkg").to_frame()
initial_budget["noramlized_Jkg"] = initial_budget["raw_Jkg"] / weight_sum

initial_budget

,raw_Jkg,noramlized_Jkg
cpT,231340.312639,261742.987484
Lvq,6558.677672,7420.617134
gh,47981.563342,54287.286077
MSE,285880.553654,323450.890695


In [13]:
weights = ds_ini.sensitivity.sum(
    dim=["height", "latitude", "longitude"],
    skipna=False
).to_series()

print(weights)

valid_time
2015-04-25 00:00:00    0.886020
2015-04-25 01:00:00    0.883845
2015-04-25 02:00:00    0.884596
2015-04-25 03:00:00    0.884517
2015-04-25 04:00:00    0.882980
2015-04-25 05:00:00    0.888528
2015-04-25 06:00:00    0.888034
2015-04-25 07:00:00    0.886708
2015-04-25 08:00:00    0.887898
2015-04-25 09:00:00    0.886478
2015-04-25 10:00:00    0.886949
2015-04-25 11:00:00    0.886559
2015-04-25 12:00:00    0.884309
2015-04-25 13:00:00    0.883916
2015-04-25 14:00:00    0.889299
2015-04-25 15:00:00    0.889935
2015-04-25 16:00:00    0.894268
2015-04-25 17:00:00    0.895989
2015-04-25 18:00:00    0.896959
2015-04-25 19:00:00    0.890129
2015-04-25 20:00:00    0.887535
2015-04-25 21:00:00    0.889690
2015-04-25 22:00:00    0.887989
2015-04-25 23:00:00    0.888687
2015-04-26 00:00:00    0.889349
Name: sensitivity, dtype: float64


In [14]:
partinit_files = sorted(run_dir.glob("partinit_*.nc"))
particle_files = sorted(run_dir.glob("partoutput_*.nc"))

print("Run", run_dir)
print("partinit-files", [p.name for p in partinit_files])
print("Number of partoutput-files", len(particle_files))

Run /jetfs/scratch/gbrack/flexpart_output/new_delhi_30d_24h_hourly_releases_bigger_domain
partinit-files ['partinit_20150526000000.nc']
Number of partoutput-files 32


In [18]:
release = 1
initial_time = pd.Timestamp(times_ini[release - 1])

with xr.open_dataset(partinit_files[0]) as partinit:
    particle_ids = partinit.particle.where(
        partinit.release == release, drop=True
    )

with xr.open_mfdataset(particle_files, combine="by_coords") as particles:
    positions = particles[["lon", "lat", "z"]].sel(
        particle=particle_ids,
        time=initial_time
    ).load()

print("Timestamp [UTC]", initial_time)
print("particles in the release", len(particle_ids))
positions

Timestamp [UTC] 2015-04-25 00:00:00
particles in the release 100000


<xarray.Dataset> Size: 2MB
Dimensions:   (particle: 100000)
Coordinates:
  * particle  (particle) int32 400kB 2400001 2400002 2400003 ... 2499999 2500000
    time      datetime64[ns] 8B 2015-04-25
Data variables:
    lon       (particle) float32 400kB -46.04 -100.0 -32.37 ... -170.2 6.596
    lat       (particle) float32 400kB 22.18 8.053 35.61 ... 31.18 55.27 15.44
    z         (particle) float32 400kB 3.052e+03 1.574e+04 ... 218.6 2.002e+03
Attributes: (12/37)
    Conventions:            CF-1.6
    title:                  FLEXPART model output
    git:                    c1ef432 Thu Oct 29 05:46:26 2026 +0100
    source:                 Version 11.0 (2023-07-11) model output
    history:                2026-11-04 10:50 +0100  created by gbrack on jet1...
    references:             Stohl et al., Atmos. Chem. Phys., 2005, doi:10.51...
    ...                     ...
    ind_source:             1
    ind_receptor:           2
    mquasilag:              0
    nested_output:          2
    sfc_only:               0
    linit_cond:             2

In [20]:
dx = abs(lon[1] - lon[0])
dy = abs(lat[1] - lat[0])

west, east = min(lon) - dx/2, max(lon) + dx/2
south, north = min(lat) - dy/2, max(lat) + dy/2

particle_lon = ((positions.lon + 180) % 360) - 180
particle_lat = positions.lat
valid = np.isfinite(particle_lon) & np.isfinite(particle_lat)

outside = {
    "western": particle_lon < west,
    "eastern": particle_lon >= east,
    "southern": particle_lat < south,
    "northern": particle_lat >= north
}

outside_any = valid & (
    outside["western"] | outside["eastern"] 
    | outside["southern"] | outside["northern"]
)
n_valid = int(valid.sum().item())
n_outside = int(outside_any.sum().item())

print(f"Outside: {n_outside:,} / {n_valid:,} valid positions "
      f"({n_outside / n_valid:.2%})")

for direction, mask in outside.items():
    print(direction, int((valid & mask).sum().item()))

Outside: 11,398 / 100,000 valid positions (11.40%)
western 8549
eastern 2409
southern 174
northern 299
